In [0]:
%run ./P5_00_Config

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
df_bronze_sales = spark.table(BRONZE_SALES)
df_bronze_holidays = spark.table(BRONZE_HOLIDAYS)
df_bronze_oil = spark.table(BRONZE_OIL)

In [0]:
df_national_sales = df_bronze_sales.groupBy("date", "family").agg(F.sum("sales").alias("sales"),F.sum("onpromotion").alias("onpromotion"))

df_holidays = df_bronze_holidays.groupBy("date").agg(F.lit(1).alias("is_holiday"))

In [0]:
df_holidays = df_bronze_holidays.groupBy("date").agg(F.lit(1).alias("is_holiday"))

df_silver = df_national_sales.join(df_holidays, on="date", how="left").join(df_bronze_oil, on="date", how="left")

In [0]:
#Fill nulls for is_holiday
df_silver = df_silver.fillna({"is_holiday": 0})

#Forward fill oil price (nulls on weekends/holidays)
window_oil = Window.orderBy("date").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_silver = df_silver.withColumn("dcoilwtico",F.last("dcoilwtico", ignorenulls=True).over(window_oil))

In [0]:
df_silver = df_silver.withColumn("year", F.year("date")) \
                     .withColumn("month", F.month("date")) \
                     .withColumn("dayofweek", F.dayofweek("date"))

df_silver = df_silver.withColumn("ingested_at", F.current_timestamp()) \
                     .withColumn("run_id", F.lit(RUN_ID))

df_silver.write.format("delta").mode("overwrite").saveAsTable(SILVER_TABLE)